# Resampling ablation: sensitivity to rho and B

This notebook reads one run of the `rho_b` ablation and shows every summary the
SI figures and table are built from. It computes nothing: produce a run with

```
python -m benchmarks.run --ablation rho_b
```

from `code/`. The rho arm sweeps the subsampling
proportion at the default resample count; the B arm sweeps the resample count
at the default proportion. The Klein case study runs through the same cells
with replicate fits; the simulations carry the ground truth.

## 1) Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from benchmarks._ablation_cells import REFERENCE_RATIO, arm_view, enumerate_cells
from benchmarks._ablation_summary import (
    HEADLINE_METRICS,
    POOLED,
    agreement_summary,
    curve_at_k_star,
    diagnostics_summary,
    rare_recall_summary,
    selection_summary,
    similarity_summary,
    spread_summary,
    study_ari_summary,
    study_selection_shares,
    table_rows,
)
from benchmarks._artifacts import read_frames
from benchmarks._registry import ABLATIONS, METRIC_DISPLAY_NAMES
from benchmarks._tables import write_ablation_table
from benchmarks._theme import metric_color, theme_context
from benchmarks.figures import figure_ablation_b, figure_ablation_rho
from benchmarks.figures._paths import REPO_ROOT

ABLATION = ABLATIONS["rho_b"]
SCALE = ABLATION.default_scale
STUDY = ABLATION.study
RHO, B = "subsample_ratio", "n_resamples"

runs = sorted((REPO_ROOT / "results" / "runs" / f"ablation_{ABLATION.name}" / SCALE).glob("*/manifest.json"))
if not runs:
    raise FileNotFoundError(f"No {SCALE} run of ablation {ABLATION.name}; run python -m benchmarks.run --ablation {ABLATION.name}")
RUN_DIR = runs[-1].parent
frames = read_frames(RUN_DIR)
rho_view = arm_view(frames, ablation=ABLATION, scale=SCALE, arm="rho")
b_view = arm_view(frames, ablation=ABLATION, scale=SCALE, arm="b")
print(RUN_DIR)
print({name: len(frame) for name, frame in frames.items()})
print("cells expected:", len(enumerate_cells(ABLATION, SCALE)), "recorded:", len(frames["cells"]))

In [ ]:
import json

manifest = json.loads((RUN_DIR / "manifest.json").read_text())
pd.Series({k: manifest[k] for k in ("run_id", "status", "scale", "config_hash", "workers", "thread_cap", "cores", "n_units", "wall_clock_s", "git_sha")})

## 2) Rho arm, simulations

Each plot below draws one line per scenario and the pooled line over all
simulated cells, against the subsampling proportion, for the two headline
selectors. Every difficulty of the rho arm is included; the hard setting is
shown on its own where rare clusters matter.

In [ ]:
def lines(ax, summary, *, x, y, metric, lo=None, hi=None, title=None, y_label=None):
    rows = summary[summary["metric_name"] == metric]
    for study, part in rows.groupby("study"):
        part = part.sort_values(x)
        if study == POOLED:
            ax.plot(part[x], part[y], color="black", linewidth=2, marker="o", label="pooled")
            if lo is not None:
                ax.fill_between(part[x], part[lo], part[hi], color="black", alpha=0.1)
        else:
            ax.plot(part[x], part[y], marker="o", markersize=3, alpha=0.7, label=study)
    ax.set_xlabel(x)
    ax.set_ylabel(y_label or y)
    ax.set_title(title or METRIC_DISPLAY_NAMES.get(metric, metric))


def pair(summary, *, x, y, lo=None, hi=None, y_label=None, logx=False):
    with theme_context():
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
        for ax, metric in zip(axes, HEADLINE_METRICS):
            lines(ax, summary, x=x, y=y, metric=metric, lo=lo, hi=hi, y_label=y_label)
            if logx:
                ax.set_xscale("log")
        axes[0].legend(fontsize=7)
        fig.tight_layout()
    return fig

In [ ]:
rho_sel = selection_summary(rho_view["selection"], x=RHO)
pair(rho_sel, x=RHO, y="recovery", lo="recovery_lo", hi="recovery_hi", y_label="k* recovery")
rho_sel[rho_sel["study"] == POOLED]

In [ ]:
pair(rho_sel, x=RHO, y="bias_mean", y_label="mean of k-hat minus k*")
pair(rho_sel, x=RHO, y="ari_mean", y_label="ARI at the selected k");

In [ ]:
rho_star = curve_at_k_star(rho_view["curves"], rho_view["datasets"], x=RHO)
pair(rho_star, x=RHO, y="value_mean", y_label="score at k*")
pair(rho_star, x=RHO, y="se_mean", y_label="reported standard error at k*");

In [ ]:
hard = ABLATION.scales[SCALE].rho_arm.difficulties[-1]
rho_recall = rare_recall_summary(rho_view["at_k"], rho_view["selection"], x=RHO, difficulty=hard)
pair(rho_recall, x=RHO, y="recall_selected", y_label=f"rare-cluster recall at k-hat ({hard})")
pair(rho_recall, x=RHO, y="recall_k_star", y_label=f"rare-cluster recall at k* ({hard})");

In [ ]:
# Score curves over k at every rho, per scenario, for the stability selector.
curves = rho_view["curves"]
sims = curves[curves["difficulty"] != ""]
with theme_context():
    scenarios = list(ABLATION.scenarios)
    fig, axes = plt.subplots(2, len(scenarios), figsize=(3.2 * len(scenarios), 6), sharey="row")
    for row, metric in enumerate(("ari_stability", "ari_generalizability")):
        for col, name in enumerate(scenarios):
            ax = axes[row, col]
            part = sims[(sims["study"] == name) & (sims["metric_name"] == metric)]
            for rho, group in part.groupby(RHO):
                mean = group.groupby("k")["metric_value"].mean()
                ax.plot(mean.index, mean.values, marker="o", markersize=3, label=f"rho={rho:g}")
            ax.set_title(f"{name}: {METRIC_DISPLAY_NAMES.get(metric, metric)}", fontsize=8)
            ax.set_xlabel("k")
    axes[0, 0].set_ylabel("stability ARI")
    axes[1, 0].set_ylabel("generalizability ARI")
    axes[0, -1].legend(fontsize=6)
    fig.tight_layout()

## 3) Rho arm, Klein

In [ ]:
klein_sel = rho_view["selection"][rho_view["selection"]["study"] == STUDY]
for metric in HEADLINE_METRICS:
    print(METRIC_DISPLAY_NAMES.get(metric, metric))
    display(study_selection_shares(klein_sel, x=RHO, metric=metric, study=STUDY))

In [ ]:
klein_ari = study_ari_summary(klein_sel, x=RHO, study=STUDY)
with theme_context():
    fig, ax = plt.subplots(figsize=(5, 3.4))
    for metric, part in klein_ari.groupby("metric_name"):
        ax.errorbar(part[RHO], part["ari_mean"], yerr=part["ari_sem"], marker="o", capsize=2, color=metric_color(metric), label=METRIC_DISPLAY_NAMES.get(metric, metric))
    ax.set_xlabel(RHO); ax.set_ylabel("ARI of selected labels to time points"); ax.legend(fontsize=7)
klein_ari

## 4) B arm, simulations and Klein

In [ ]:
b_agree = agreement_summary(b_view["selection"], x=B)
pair(b_agree, x=B, y="agreement", y_label="replicate agreement on the selection", logx=True)
b_agree[b_agree["study"].isin([POOLED, STUDY])]

In [ ]:
b_spread = spread_summary(b_view["curves"], x=B)
pair(b_spread, x=B, y="spread", y_label="SD of the score over replicates", logx=True)
b_star = curve_at_k_star(b_view["curves"], b_view["datasets"], x=B)
pair(b_star, x=B, y="se_mean", y_label="reported standard error at k*", logx=True);

In [ ]:
b_sel = selection_summary(b_view["selection"], x=B)
pair(b_sel, x=B, y="recovery", lo="recovery_lo", hi="recovery_hi", y_label="k* recovery", logx=True)
pair(b_sel, x=B, y="bias_mean", y_label="mean of k-hat minus k*", logx=True)
b_sel[b_sel["study"] == POOLED]

In [ ]:
klein_b = b_view["selection"][b_view["selection"]["study"] == STUDY]
for metric in HEADLINE_METRICS:
    print(METRIC_DISPLAY_NAMES.get(metric, metric))
    display(study_selection_shares(klein_b, x=B, metric=metric, study=STUDY))

## 5) Subsample versus full-data similarity

ARI between a clustering of a subsample and the full-data clustering
restricted to that subsample, per scenario and candidate k. The point at
ratio 1 is the refit reference: the full data clustered again with another
seed, which separates the estimator's own randomness from subsampling.

In [ ]:
sim = similarity_summary(rho_view["similarity"])
with theme_context():
    studies = [*ABLATION.scenarios, STUDY]
    fig, axes = plt.subplots(1, len(studies), figsize=(3.0 * len(studies), 3.2), sharey=True)
    for ax, name in zip(axes, studies):
        part = sim[sim["study"] == name]
        for (estimator, k), group in part.groupby(["estimator", "k"]):
            group = group.sort_values(RHO)
            ax.plot(group[RHO], group["ari_mean"], marker="o", markersize=3, label=f"{estimator[:5]} k={k}")
        ax.axvline(REFERENCE_RATIO, color="grey", linestyle=":")
        ax.set_title(name, fontsize=8); ax.set_xlabel(RHO)
    axes[0].set_ylabel("ARI to full-data fit"); axes[-1].legend(fontsize=5)
    fig.tight_layout()
sim[sim["study"] == STUDY]

## 6) Diagnostics

Per setting and study: mean fit time, the largest fraction of never-co-sampled
entries in the stability consensus (training pairs) and in the generalizability
consensus (test-set pairs; sparser at the grid ends, and the matrix the
generalizability-mode labels are cut from), and the mean count of
cluster-count warnings.

In [ ]:
diag_rho = diagnostics_summary(rho_view["cells"], x=RHO)
diag_b = diagnostics_summary(b_view["cells"], x=B)
with theme_context():
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    for row, (diag, x) in enumerate(((diag_rho, RHO), (diag_b, B))):
        for col, y in enumerate((
            "fit_seconds",
            "consensus_nan_fraction",
            "consensus_generalizability_nan_fraction",
            "n_cluster_count_warnings",
        )):
            ax = axes[row, col]
            for study, part in diag.groupby("study"):
                part = part.sort_values(x)
                ax.plot(part[x], part[y], marker="o", markersize=3, label=study)
            ax.set_xlabel(x); ax.set_ylabel(y)
            if x == B:
                ax.set_xscale("log")
    axes[0, 0].legend(fontsize=6)
    fig.tight_layout()
pd.concat([diag_rho.assign(arm="rho"), diag_b.assign(arm="b")])

## 7) SI figures and table

Written under `vis/benchmarking/` and `results/tables/`; copying into the
manuscript stays a manual step.

In [ ]:
figure_ablation_rho(frames, ablation=ABLATION, scale=SCALE)
figure_ablation_b(frames, ablation=ABLATION, scale=SCALE)
table_path = write_ablation_table(frames, ablation=ABLATION, scale=SCALE, out_dir=REPO_ROOT / "results" / "tables")
print(table_path.read_text())

In [ ]:
table_rows(rho_view, x=RHO, study=STUDY)

In [ ]:
table_rows(b_view, x=B, study=STUDY)